# TrackRAG-Gulf: real GPU grounding pilot

This notebook prepares a **two-step QLoRA update, save and fresh-process reload** for the planned Qwen3-VL-8B model. It has not been executed with the actual 8B model on a cloud GPU yet. Passing local software checks or source-data validation is not evidence that this GPU run fits or succeeds.

The ten public examples supervise destination geometry only. This pilot does not complete the final local instruction/ambiguity corpus, held-out evaluation, voice interaction, or the project.

Use a disposable Google Colab **GPU** runtime. GPU availability is controlled by your account; this notebook does not provision or purchase one. The initial checks require at least 12 GiB total GPU memory, 10 GiB free GPU memory and 30 GiB free disk. These are screening floors, not a fit guarantee. It will stop on a 4 GB laptop GPU or CPU runtime.

No private GitHub login or token is required: upload the minimal source ZIP built on your laptop. Model/data downloads happen on the cloud runtime, not on C: or D:.

## Before opening the GPU session

From the current project checkout, create a new minimal bundle:

```text
python -m tools.dropoff_bundle source --out runs/dropoff-cloud-source.zip
```

Keep the printed SHA256. Open this notebook in Colab, choose a GPU through **Runtime → Change runtime type**, then run the cells in order. Upload the source ZIP when requested. The ZIP must be one you built from this project; its internal hashes detect alteration, but are not an independent publisher signature.

The notebook keeps installation, caches and training artifacts under a new `/content` workspace. It does not upload your whole checkout, virtual environments, credentials, recordings or existing model caches.

In [ ]:
import csv
from datetime import datetime, timezone
import io
import json
import os
from pathlib import Path
import shutil
import subprocess
import sys
import uuid

if not Path('/content').is_dir():
    raise RuntimeError('Run this notebook in a Google Colab GPU runtime; it is not a laptop training command.')
try:
    report = subprocess.run(
        ['nvidia-smi', '--query-gpu=index,name,memory.total,memory.free', '--format=csv,noheader,nounits'],
        check=True, capture_output=True, text=True,
    )
except (FileNotFoundError, subprocess.CalledProcessError) as error:
    raise RuntimeError('No working NVIDIA GPU was detected. Select a GPU runtime before installing/downloading models.') from error
rows = list(csv.reader(io.StringIO(report.stdout)))
if not rows:
    raise RuntimeError('No NVIDIA GPU was reported.')
index, name, total_mib, free_mib = [item.strip() for item in rows[0]]
total_gib, free_gib = float(total_mib) / 1024, float(free_mib) / 1024
disk_gib = shutil.disk_usage('/content').free / 2**30
print(json.dumps({'gpu_index': index, 'gpu': name, 'total_gib': total_gib,
                  'free_gib': free_gib, 'disk_free_gib': disk_gib, 'fit_proven': False}, indent=2))
if total_gib < 12 or free_gib < 10 or disk_gib < 30:
    raise RuntimeError('Insufficient screening resources: need GPU >=12 GiB total, >=10 GiB free and disk >=30 GiB. No model download has started.')
WORK_ROOT = Path('/content') / ('trackrag-grounding-' + datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S') + '-' + uuid.uuid4().hex[:8])
WORK_ROOT.mkdir()
PROJECT = WORK_ROOT / 'project'
VENV = WORK_ROOT / 'training-venv'
CACHE = WORK_ROOT / 'model-cache'
RUN = WORK_ROOT / 'runs' / 'pilot'
CLOUD_ENV = dict(os.environ)
CLOUD_ENV.update({'HF_HOME': str(CACHE), 'HUGGINGFACE_HUB_CACHE': str(CACHE / 'hub'),
                  'PIP_CACHE_DIR': str(WORK_ROOT / 'pip-cache'), 'TMPDIR': str(WORK_ROOT / 'tmp')})
Path(CLOUD_ENV['TMPDIR']).mkdir()
print('Cloud workspace:', WORK_ROOT)


## Validate the uploaded source before extracting it

The following helper accepts only the minimal source paths, rejects traversal/symlink/duplicate entries, and checks every declared size and SHA256. It validates the whole archive before creating the project directory.

In [ ]:
import hashlib
import io
import json
from pathlib import Path, PurePosixPath
import re
import stat
import zipfile

SOURCE_REQUIRED = {'trackrag/dropoff/data.py', 'trackrag/dropoff/contracts.py', 'trackrag/dropoff/formatting.py', 'trackrag/dropoff/__init__.py', 'tools/dropoff_train.py', 'configs/dropoff/regseg-scene-guards.json', 'trackrag/dropoff/model.py', 'configs/dropoff/training-pilot.json', 'requirements-grounding-train.txt', 'tools/dropoff_bootstrap_regseg.py', 'trackrag/contracts.py', 'tools/dropoff_bundle.py'}

def verify_and_extract_source_bundle(blob, destination):
    limit = 64 * 1024 * 1024
    if not isinstance(blob, bytes) or len(blob) > limit:
        raise ValueError('Uploaded source ZIP exceeds its bounded size.')
    def unique(pairs):
        value = {}
        for key, item in pairs:
            if key in value:
                raise ValueError('Duplicate manifest field')
            value[key] = item
        return value
    def invalid_constant(value):
        raise ValueError('Nonfinite JSON constant')
    def safe(name):
        value = PurePosixPath(name)
        if (not name or '\\' in name or ':' in name or '\x00' in name or value.is_absolute()
                or '..' in value.parts or name != value.as_posix()):
            raise ValueError('Unsafe source ZIP path')
        return value
    def allowed(name):
        value = safe(name)
        parts = value.parts
        forbidden = {'__pycache__', '.git', '.venv', '.venv-models', 'venv', 'env', 'data', 'models', 'runs'}
        if any(part.startswith('.') or part in forbidden for part in parts):
            return False
        if name in ('trackrag/__init__.py', 'trackrag/contracts.py', 'requirements-grounding-train.txt', 'requirements-grounding-local.txt', 'experiments/dropoff/public-contrast-protocol-20261121.json'):
            return True
        if len(parts) >= 3 and parts[:2] == ('trackrag', 'dropoff'):
            return value.suffix == '.py' and '__pycache__' not in parts
        if len(parts) == 2 and parts[0] == 'tools':
            return parts[1].startswith('dropoff_') and value.suffix == '.py'
        if len(parts) == 3 and parts[:2] == ('configs', 'dropoff'):
            return value.suffix == '.json'
        if len(parts) == 2 and parts[0] == 'docs':
            return parts[1].startswith('dropoff') and value.suffix == '.md'
        if len(parts) == 2 and parts[0] == 'tests':
            return parts[1].startswith('test_dropoff') and value.suffix == '.py'
        return False
    with zipfile.ZipFile(io.BytesIO(blob)) as archive:
        entries = archive.infolist()
        names = [entry.filename for entry in entries]
        if len(names) > 1024 or len(set(names)) != len(names) or len({name.casefold() for name in names}) != len(names):
            raise ValueError('Duplicate/colliding or excessive ZIP entries')
        for entry in entries:
            safe(entry.filename)
            mode = entry.external_attr >> 16
            if (entry.orig_filename != entry.filename or entry.is_dir() or stat.S_ISLNK(mode)
                    or stat.S_IFMT(mode) not in (0, stat.S_IFREG) or entry.flag_bits & 1):
                raise ValueError('Source ZIP contains a non-regular or encrypted entry')
        manifest_name = 'BUNDLE_MANIFEST.json'
        if manifest_name not in names or archive.getinfo(manifest_name).file_size > 1024 * 1024:
            raise ValueError('Source manifest is missing or oversized')
        manifest_blob = archive.read(manifest_name)
        manifest = json.loads(manifest_blob.decode('utf-8'), object_pairs_hook=unique, parse_constant=invalid_constant)
        if (not isinstance(manifest, dict) or set(manifest) != {'schema_version', 'bundle_kind', 'files', 'metadata'}
                or type(manifest['schema_version']) is not int or manifest['schema_version'] != 1
                or manifest['bundle_kind'] != 'source'):
            raise ValueError('Unsupported source manifest')
        declared = manifest['files']
        if (not isinstance(declared, dict) or not declared or set(names) != set(declared) | {manifest_name}
                or not SOURCE_REQUIRED <= set(declared)):
            raise ValueError('Source contents differ from the manifest or required files are missing')
        if sum(entry.file_size for entry in entries) > limit + 1024 * 1024:
            raise ValueError('Expanded source ZIP exceeds its limit')
        for name in declared:
            if any(parent.as_posix() in declared for parent in PurePosixPath(name).parents if parent.as_posix() != '.'):
                raise ValueError('Conflicting file/directory source paths')
        contents = {}
        for name, expected in declared.items():
            if (not allowed(name) or not isinstance(expected, dict) or set(expected) != {'bytes', 'sha256'}
                    or type(expected['bytes']) is not int or expected['bytes'] < 0
                    or not isinstance(expected['sha256'], str) or not re.fullmatch('[0-9a-f]{64}', expected['sha256'])):
                raise ValueError('Invalid source path or declared hash/size')
            if archive.getinfo(name).file_size != expected['bytes']:
                raise ValueError('Source file size differs from its manifest')
            content = archive.read(name)
            if len(content) != expected['bytes'] or hashlib.sha256(content).hexdigest() != expected['sha256']:
                raise ValueError('Source file SHA256 differs from its manifest')
            contents[name] = content
    destination = Path(destination).resolve()
    if destination.exists():
        raise FileExistsError('Choose a fresh project extraction directory')
    destination.mkdir(parents=True)
    for name, content in {**contents, 'BUNDLE_MANIFEST.json': manifest_blob}.items():
        target = destination / name
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(content)
    return {'files_verified': len(contents), 'bundle_sha256': hashlib.sha256(blob).hexdigest(),
            'model_weights_included': False, 'project': str(destination)}


In [ ]:
from google.colab import files

uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError('Upload exactly one locally built dropoff source ZIP.')
source_name, source_bytes = next(iter(uploaded.items()))
if not source_name.lower().endswith('.zip'):
    raise ValueError('Expected the source ZIP, not model weights or a dataset archive.')
source_receipt = verify_and_extract_source_bundle(source_bytes, PROJECT)
(WORK_ROOT / 'uploaded-source.zip').write_bytes(source_bytes)
print(json.dumps(source_receipt, indent=2))
print('Compare this bundle SHA256 with the value printed by the local bundler.')


## Install the separate cloud training environment

This installs the supplied training requirements into a new cloud virtual environment. It does not modify your laptop reviewer environment. Installation and model caches remain in this cloud workspace; `--no-cache-dir` avoids keeping a second Torch wheel cache. Exact resolved versions are saved with the pilot artifacts.

A pinned lightweight virtualenv fallback is used only if the cloud Python lacks standard-library venv/ensurepip support.

In [ ]:
try:
    subprocess.run([sys.executable, '-m', 'venv', str(VENV)], check=True, env=CLOUD_ENV)
except subprocess.CalledProcessError:
    bootstrap_tools = WORK_ROOT / 'bootstrap-tools'
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check', '--no-cache-dir',
                    '--target', str(bootstrap_tools), 'virtualenv==20.35.3'], check=True, env=CLOUD_ENV)
    bootstrap_env = dict(CLOUD_ENV, PYTHONPATH=str(bootstrap_tools))
    subprocess.run([sys.executable, '-m', 'virtualenv', '--no-download',
                    '--app-data', str(WORK_ROOT / 'virtualenv-app-data'), str(VENV)],
                   check=True, env=bootstrap_env)
PYTHON = VENV / 'bin' / 'python'
subprocess.run([str(PYTHON), '-m', 'pip', 'install', '--disable-pip-version-check', '--no-cache-dir',
                '-r', str(PROJECT / 'requirements-grounding-train.txt')], check=True, env=CLOUD_ENV)
freeze = subprocess.run([str(PYTHON), '-m', 'pip', 'freeze'], check=True, capture_output=True, text=True, env=CLOUD_ENV)
ENV_FREEZE = WORK_ROOT / 'environment_freeze.txt'
ENV_FREEZE.write_text(freeze.stdout, encoding='utf-8')
print('Installed separate cloud environment; resolved package versions saved:', ENV_FREEZE)


## Download ten genuine training examples

The bootstrap fetches bounded public metadata, a 4 MiB image prefix and the small mask archive member. It verifies source hashes and excludes entire training scenes reused by the concise public test. It does not use concise-test or validation images for gradient updates. It does not invent ambiguity/anchor labels for mask-only public data.

In [ ]:
DATA = PROJECT / 'data' / 'pilot'
CONFIG = PROJECT / 'configs' / 'dropoff' / 'training-pilot.json'
subprocess.run([str(PYTHON), '-m', 'tools.dropoff_bootstrap_regseg', '--out', str(DATA), '--limit', '10',
                '--image-prefix-mib', '4', '--max-download-mib', '128'],
               cwd=PROJECT, env=CLOUD_ENV, check=True)
CASES = DATA / 'cases.jsonl'
subprocess.run([str(PYTHON), '-m', 'tools.dropoff_train', 'validate', '--config', str(CONFIG),
                '--train', str(CASES)], cwd=PROJECT, env=CLOUD_ENV, check=True)


## Check the installed stack before model weights are downloaded

This is separate from the output directory used for training. A passing preflight still does not prove that the 8B model fits or trains numerically on the allocated GPU.

In [ ]:
preflight = subprocess.run([str(PYTHON), '-m', 'tools.dropoff_train', 'preflight', '--config', str(CONFIG),
                             '--cache-dir', str(CACHE), '--out', str(WORK_ROOT / 'preflight-preview')],
                            cwd=PROJECT, env=CLOUD_ENV, capture_output=True, text=True)
print(preflight.stdout)
if preflight.stderr:
    print(preflight.stderr)
PREFLIGHT_FILE = WORK_ROOT / 'preflight.json'
PREFLIGHT_FILE.write_text(preflight.stdout, encoding='utf-8')
if preflight.returncode != 0:
    raise RuntimeError('Training preflight is blocked. No smaller model is substituted; inspect the report before proceeding.')


## Perform the actual two-step pilot

This is the first cell that downloads the pinned 8B base weights. It performs real LoRA parameter updates, saves the adapter and checks a fresh-base reload. The generation probe uses a training image and is a save/reload smoke test, not held-out accuracy. If memory, dependencies or spatial generation fail, retain the failure artifacts and use the diagnostic download cell below; do not describe that run as completed.

In [ ]:
if RUN.exists():
    raise FileExistsError('Choose a fresh training run directory rather than overwrite previous evidence.')
SESSION_HANDOFF = WORK_ROOT / 'cloud_handoff.json'
handoff = {
    'source_bundle_sha256': source_receipt['bundle_sha256'], 'training_exit_code': None,
    'gpu': name, 'gpu_total_gib_at_start': total_gib, 'pilot_only': True,
    'held_out_quality_verified': False,
    'fresh_process_reload': {'status': 'not_started', 'exit_code': None, 'verified': False},
}
SESSION_HANDOFF.write_text(json.dumps(handoff, indent=2) + '\n', encoding='utf-8')
try:
    training = subprocess.run([str(PYTHON), '-m', 'tools.dropoff_train', 'train', '--config', str(CONFIG),
                               '--train', str(CASES), '--cache-dir', str(CACHE), '--out', str(RUN)],
                              cwd=PROJECT, env=CLOUD_ENV)
    TRAIN_EXIT = training.returncode
    handoff['training_exit_code'] = TRAIN_EXIT
except BaseException as error:
    handoff['training_exception'] = {'type': type(error).__name__, 'message': str(error)}
    raise
finally:
    SESSION_HANDOFF.write_text(json.dumps(handoff, indent=2) + '\n', encoding='utf-8')
    if RUN.is_dir():
        shutil.copyfile(SESSION_HANDOFF, RUN / 'cloud_handoff.json')
        if ENV_FREEZE.is_file():
            shutil.copyfile(ENV_FREEZE, RUN / 'environment_freeze.txt')
if TRAIN_EXIT != 0:
    if (RUN / 'failure.json').is_file():
        print((RUN / 'failure.json').read_text())
    raise RuntimeError('The real pilot failed or did not complete a valid spatial probe. Export labelled diagnostics below.')


## Verify saved parameters in a fresh Python process

This reuses the already cached base weights and the portable public probe image saved with the run. It is a separate-process handoff check, not a second training run or a quality benchmark.

In [ ]:
from datetime import datetime, timezone
import json
from pathlib import Path
import subprocess

def run_fresh_process_reload(command, run_dir, *, cwd, env, runner=None):
    runner = runner or subprocess.run
    run_dir = Path(run_dir)
    handoff_path = run_dir / 'cloud_handoff.json'
    handoff = json.loads(handoff_path.read_text()) if handoff_path.is_file() else {'training_exit_code': None}
    attempt = {'status': 'running', 'exit_code': None, 'verified': False,
               'started_at': datetime.now(timezone.utc).isoformat()}
    handoff['fresh_process_reload'] = attempt
    handoff_path.write_text(json.dumps(handoff, indent=2) + '\n', encoding='utf-8')
    (run_dir / 'cloud_reload_stdout.txt').write_text('', encoding='utf-8')
    (run_dir / 'cloud_reload_stderr.txt').write_text('', encoding='utf-8')
    def as_text(value):
        return value.decode('utf-8', errors='replace') if isinstance(value, bytes) else str(value or '')
    try:
        result = runner(command, cwd=cwd, env=env, capture_output=True, text=True, timeout=1800)
        attempt['exit_code'] = result.returncode
        (run_dir / 'cloud_reload_stdout.txt').write_text(as_text(result.stdout), encoding='utf-8')
        (run_dir / 'cloud_reload_stderr.txt').write_text(as_text(result.stderr), encoding='utf-8')
        print(as_text(result.stdout))
        if result.stderr:
            print(as_text(result.stderr))
        if result.returncode != 0:
            raise RuntimeError('Fresh-process reload exited with code ' + str(result.returncode))
        receipt = json.loads((run_dir / 'reload_verification.json').read_text())
        if receipt.get('verified') is not True:
            raise RuntimeError('Fresh-process spatial-generation receipt is not verified.')
        attempt.update(status='passed', verified=True)
    except BaseException as error:
        attempt.update(status='failed', verified=False, error_type=type(error).__name__, error_message=str(error))
        for attribute, filename in (('stdout', 'cloud_reload_stdout.txt'), ('stderr', 'cloud_reload_stderr.txt')):
            value = getattr(error, attribute, None)
            if value is not None:
                (run_dir / filename).write_text(as_text(value), encoding='utf-8')
        stderr_path = run_dir / 'cloud_reload_stderr.txt'
        if not stderr_path.read_text(encoding='utf-8'):
            stderr_path.write_text(type(error).__name__ + ': ' + str(error) + '\n', encoding='utf-8')
        raise
    finally:
        attempt['finished_at'] = datetime.now(timezone.utc).isoformat()
        handoff_path.write_text(json.dumps(handoff, indent=2) + '\n', encoding='utf-8')
    return receipt

if globals().get('TRAIN_EXIT') != 0:
    raise RuntimeError('Training has not succeeded; use the diagnostic export cell.')
reload_receipt = run_fresh_process_reload(
    [str(PYTHON), '-m', 'tools.dropoff_train', 'verify-reload', '--run-dir', str(RUN), '--cache-dir', str(CACHE)],
    RUN, cwd=PROJECT, env=CLOUD_ENV,
)
print(json.dumps(reload_receipt, indent=2))
print('Save/reload pilot completed. Held-out grounding quality and final project completion remain unverified.')


## Download the adapter and evidence, or labelled diagnostics

The download excludes base-model weights, caches and bulk datasets. It includes the adapter, configurations, parameter-update/reload receipts, logs and one portable public training probe image. If the run failed, the bundle is explicitly labelled diagnostic and cannot masquerade as a successful trained artifact. Keep the source ZIP too.

Download before the Colab runtime disconnects. The two-step pilot must be followed by training on the reviewed full corpus with separate development data, integration and held-out evaluation.

In [ ]:
if RUN.is_dir():
    artifact_dir = RUN
    if ENV_FREEZE.is_file():
        shutil.copyfile(ENV_FREEZE, artifact_dir / 'environment_freeze.txt')
else:
    artifact_dir = WORK_ROOT / 'preflight-diagnostics'
    artifact_dir.mkdir(exist_ok=True)
    if globals().get('PREFLIGHT_FILE') is not None and PREFLIGHT_FILE.is_file():
        shutil.copyfile(PREFLIGHT_FILE, artifact_dir / 'preflight.json')
    if globals().get('ENV_FREEZE') is not None and ENV_FREEZE.is_file():
        shutil.copyfile(ENV_FREEZE, artifact_dir / 'environment_freeze.txt')
# A cloud export always carries the session outcome. Missing/pending/failed
# fresh-process verification cannot fall back to stale inner-training flags.
if not (artifact_dir / 'cloud_handoff.json').is_file():
    session_file = globals().get('SESSION_HANDOFF')
    if session_file is not None and session_file.is_file():
        shutil.copyfile(session_file, artifact_dir / 'cloud_handoff.json')
    else:
        (artifact_dir / 'cloud_handoff.json').write_text(json.dumps({
            'training_exit_code': None, 'training_run_created': RUN.is_dir(),
            'fresh_process_reload': {'status': 'not_started', 'exit_code': None, 'verified': False},
        }, indent=2) + '\n', encoding='utf-8')
OUTPUT_ZIP = WORK_ROOT / ('grounding-pilot-handoff-' + uuid.uuid4().hex[:8] + '.zip')
# Always permit labelled diagnostics; the bundler itself decides whether all
# training, hash, schema and fresh-process completion requirements passed.
packaging = subprocess.run(
    [str(PYTHON), '-m', 'tools.dropoff_bundle', 'artifacts', '--run-dir', str(artifact_dir),
     '--out', str(OUTPUT_ZIP), '--allow-incomplete'],
    cwd=PROJECT, env=CLOUD_ENV, check=True, capture_output=True, text=True,
)
bundle_result = json.loads(packaging.stdout)
print(json.dumps(bundle_result, indent=2))
label = 'adapter' if bundle_result['bundle_kind'] == 'adapter' else 'diagnostics'
labelled_zip = OUTPUT_ZIP.with_name(OUTPUT_ZIP.name.replace('-handoff-', '-' + label + '-'))
if labelled_zip.exists():
    raise FileExistsError('Choose a fresh handoff filename rather than overwrite an existing download.')
OUTPUT_ZIP.rename(labelled_zip)
OUTPUT_ZIP = labelled_zip
subprocess.run([str(PYTHON), '-m', 'tools.dropoff_bundle', 'verify', '--bundle', str(OUTPUT_ZIP)],
               cwd=PROJECT, env=CLOUD_ENV, check=True)
print('Download:', OUTPUT_ZIP.name, '; type:', bundle_result['bundle_kind'])
files.download(str(OUTPUT_ZIP))
